In [1]:
from __future__ import annotations

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from retrieval import MIN_SCORE, TOP_K, load_chunks

In [2]:
## labelled query set: Ground Truth ##

COVERED: dict[str, int] = {
    "how long do I have to submit an expense claim": 14,
    "what is the per diem for meals on a trip": 15,
    "who has to approve international business travel": 11,
    "can I book business class on a long flight": 13,
    "am I insured while travelling abroad": 12,
    "what is the hotel cap for domestic travel": 10,
    "how many days of annual leave do I get": 5,
    "can I carry unused leave into next year": 6,
    "how much paid sick leave is there": 7,
    "what happens if I am sick for more than two weeks": 8,
    "can I work from another country for a while": 4,
    "which days must I be in the office": 3,
    "taking a work laptop to a restricted country": 19,
    "when must I report a stolen laptop": 18,
    "how often must I reconcile my corporate card": 17,
    "who is issued a corporate credit card": 16,
    "do I repay training costs if I resign": 28,
    "what is my annual learning budget": 27,
    "can I accept a gift from a supplier": 29,
    "what response time do Enterprise customers get": 33,
    "what counts as a P1 incident": 34,
    "can a new customer cancel for a refund": 36,
    "what editions is the platform sold in": 32,
    "how is information classified": 21,
    "when is probation confirmed": 23,
}

In [3]:
class TfidfScorer:
    """Cosine similarity over TF-IDF vectors."""

    name = "TF-IDF (cosine)"

    def __init__(self, chunks: list[str]) -> None:
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform(chunks)

    def scores(self, query: str) -> np.ndarray:
        return cosine_similarity(self.vectorizer.transform([query]), self.matrix)[0]


class Bm25Scorer:
    def __init__(self, chunks: list[str], k1: float = 1.5, b: float = 0.75) -> None:
        self.k1, self.b = k1, b
        self.name = f"BM25 (k1={k1}, b={b})"
        self.vectorizer = CountVectorizer(stop_words="english")
        self.tf = self.vectorizer.fit_transform(chunks).toarray().astype(float)
        n_docs = self.tf.shape[0]
        doc_freq = (self.tf > 0).sum(axis=0)
        self.idf = np.log(1 + (n_docs - doc_freq + 0.5) / (doc_freq + 0.5))
        self.doc_len = self.tf.sum(axis=1)
        self.avg_len = self.doc_len.mean()
        self.analyzer = self.vectorizer.build_analyzer()

    def scores(self, query: str) -> np.ndarray:
        terms = [
            self.vectorizer.vocabulary_[t]
            for t in self.analyzer(query)
            if t in self.vectorizer.vocabulary_
        ]
        if not terms:
            return np.zeros(self.tf.shape[0])
        freq = self.tf[:, terms]
        numerator = freq * (self.k1 + 1)
        denominator = freq + self.k1 * (
            1 - self.b + self.b * (self.doc_len / self.avg_len)
        )[:, None]
        return (self.idf[terms] * (numerator / denominator)).sum(axis=1)

In [4]:
def rank_of(scores: np.ndarray, gold: int) -> int:
    """1-indexed position of the gold chunk in the ranked results."""
    return int(np.argsort(scores)[::-1].tolist().index(gold)) + 1


def ranking_report(scorer) -> dict[str, float]:
    """
    Recall@K  -- was the right chunk anywhere in the top K? (K = TOP_K, what the
                 retriever actually passes on, so this is the metric that matters)
    MRR       -- average of 1/rank; partial credit for being near the top
    Precision@1 -- was the right chunk first?
    """
    in_top_k, reciprocal_ranks, first = 0, 0.0, 0
    for query, gold in COVERED.items():
        rank = rank_of(scorer.scores(query), gold)
        in_top_k += rank <= TOP_K
        reciprocal_ranks += 1 / rank
        first += rank == 1
    n = len(COVERED)
    return {
        f"Recall@{TOP_K}": in_top_k / n,
        "MRR": reciprocal_ranks / n,
        "Precision@1": first / n,
    }


def length_stats(chunks: list[str]) -> tuple[float, int, int]:
    """Coefficient of variation of chunk length (std/mean), plus the range."""
    counts = CountVectorizer(stop_words="english").fit_transform(chunks).toarray()
    lengths = counts.sum(axis=1)
    return lengths.std() / lengths.mean(), int(lengths.min()), int(lengths.max())


# Filler with no topical overlap with the query set: changes chunk length
# without changing which chunk answers which question.
_BOILERPLATE = (
    " This provision is issued under the governance framework and remains subject to "
    "periodic review by the responsible committee. Queries regarding interpretation "
    "should be directed through the standard escalation route. Nothing in this "
    "provision limits any obligation arising elsewhere in this document. "
) * 6


def _row(label: str, report: dict[str, float]) -> str:
    return (
        f"{label:<24}{report[f'Recall@{TOP_K}']:>12.3f}"
        f"{report['MRR']:>10.3f}{report['Precision@1']:>14.3f}"
    )

In [5]:
chunks = load_chunks()
cv, shortest, longest = length_stats(chunks)

print(f"\nCorpus     : {len(chunks)} chunks, {len(COVERED)} labelled questions")
print(f"Chunk length: {shortest}-{longest} terms, CV {cv:.3f}")

print(f"\n{'='*66}\n1. TUNING BM25 — which k1 and b to compare against\n{'='*66}")
print("  Comparing against one arbitrary setting would prove nothing, so both")
print("  parameters are swept first and the best is carried into section 2.")
print("    k1 -- how fast term frequency saturates")
print("    b  -- how strongly long chunks are penalised (0 = none, 1 = full)\n")

grid = [
    (k1, b, ranking_report(Bm25Scorer(chunks, k1=k1, b=b)))
    for k1 in (0.9, 1.2, 1.5, 2.0)
    for b in (0.0, 0.25, 0.5, 0.75, 1.0)
]
print(f"   {'k1':>5}{'b':>7}{'Recall@' + str(TOP_K):>12}{'MRR':>9}{'P@1':>9}")
previous_k1 = None
for k1, b, report in grid:
    if previous_k1 is not None and k1 != previous_k1:
        print()
    previous_k1 = k1
    print(
        f"   {k1:>5}{b:>7}{report[f'Recall@{TOP_K}']:>12.3f}"
        f"{report['MRR']:>9.3f}{report['Precision@1']:>9.3f}"
    )

best_mrr = max(report["MRR"] for _, _, report in grid)
best = [(k1, b) for k1, b, report in grid if abs(report["MRR"] - best_mrr) < 1e-9]
best_k1, best_b = best[0]
print(f"\n  Best MRR {best_mrr:.3f}, reached by {len(best)} of {len(grid)} settings.")
print(f"  Section 2 uses k1={best_k1}, b={best_b} -- BM25 at its best on this corpus.")

print(f"\n{'='*66}\n2. RANKING QUALITY\n{'='*66}")
print(f"{'':24}{'Recall@' + str(TOP_K):>12}{'MRR':>10}{'Precision@1':>14}")
for scorer in (TfidfScorer(chunks), Bm25Scorer(chunks, k1=best_k1, b=best_b)):
    print(_row(scorer.name, ranking_report(scorer)))

print(f"\n{'='*66}\n3. WHY — ranking quality vs chunk-length variance\n{'='*66}")
print("  BM25's advantage over TF-IDF is the b parameter tuned above, and it")
print("  only earns its keep when chunk lengths vary. Padding chunks with")
print("  topically neutral boilerplate raises the variance without changing")
print("  relevance or ground truth, which isolates that one effect.\n")

print(f"{'corpus':<34}{'CV':>7}{'TF-IDF MRR':>13}{'BM25 MRR':>11}{'gap':>8}")
variants = [
    ("as shipped", chunks),
    ("half the chunks padded", [
        c + _BOILERPLATE if i % 2 == 0 else c for i, c in enumerate(chunks)
    ]),
    ("every third padded 3x", [
        c + _BOILERPLATE * 3 if i % 3 == 0 else c for i, c in enumerate(chunks)
    ]),
]
for label, variant in variants:
    variant_cv, _, _ = length_stats(variant)
    tf_mrr = ranking_report(TfidfScorer(variant))["MRR"]
    bm_mrr = ranking_report(Bm25Scorer(variant, k1=best_k1, b=best_b))["MRR"]
    print(
        f"{label:<34}{variant_cv:>7.3f}{tf_mrr:>13.3f}"
        f"{bm_mrr:>11.3f}{bm_mrr - tf_mrr:>8.3f}"
    )


Corpus     : 37 chunks, 25 labelled questions
Chunk length: 18-76 terms, CV 0.264

1. TUNING BM25 — which k1 and b to compare against
  Comparing against one arbitrary setting would prove nothing, so both
  parameters are swept first and the best is carried into section 2.
    k1 -- how fast term frequency saturates
    b  -- how strongly long chunks are penalised (0 = none, 1 = full)

      k1      b    Recall@5      MRR      P@1
     0.9    0.0       0.960    0.835    0.720
     0.9   0.25       0.960    0.835    0.720
     0.9    0.5       0.960    0.835    0.720
     0.9   0.75       0.960    0.835    0.720
     0.9    1.0       0.960    0.835    0.720

     1.2    0.0       0.960    0.835    0.720
     1.2   0.25       0.960    0.835    0.720
     1.2    0.5       0.960    0.841    0.720
     1.2   0.75       0.960    0.841    0.720
     1.2    1.0       0.960    0.835    0.720

     1.5    0.0       0.960    0.835    0.720
     1.5   0.25       0.960    0.835    0.720
     1.5  